## 1. Load CSV Data

In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import optuna

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics       import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble      import RandomForestClassifier

DATA_DIR = 'data/'
DATA_PVS = 'PVS 1'

labels_path = os.path.join(DATA_DIR, DATA_PVS,'dataset_labels.csv')
labels = pd.read_csv(labels_path)

gps_path = os.path.join(DATA_DIR, DATA_PVS,'dataset_gps.csv')
gps = pd.read_csv(gps_path)

left_path = os.path.join(DATA_DIR, DATA_PVS,'dataset_gps_mpu_left.csv')
right_path = os.path.join(DATA_DIR, DATA_PVS,'dataset_gps_mpu_right.csv')

left = pd.read_csv(left_path)
right = pd.read_csv(right_path)

left_settings_path = os.path.join(DATA_DIR, DATA_PVS,'dataset_settings_left.csv')
right_settings_path = os.path.join(DATA_DIR, DATA_PVS,'dataset_settings_right.csv')

left_settings = pd.read_csv(left_settings_path)
right_settings = pd.read_csv(right_settings_path)


display(labels.head())

,paved_road,unpaved_road,dirt_road,cobblestone_road,asphalt_road,no_speed_bump,speed_bump_asphalt,speed_bump_cobblestone,good_road_left,regular_road_left,bad_road_left,good_road_right,regular_road_right,bad_road_right
0,1,0,0,0,1,1,0,0,1,0,0,1,0,0
1,1,0,0,0,1,1,0,0,1,0,0,1,0,0
2,1,0,0,0,1,1,0,0,1,0,0,1,0,0
3,1,0,0,0,1,1,0,0,1,0,0,1,0,0
4,1,0,0,0,1,1,0,0,1,0,0,1,0,0


## 2. Decode One-Hot Labels into Categorical Columns

In [36]:
surface_cols   = ['dirt_road', 'cobblestone_road', 'asphalt_road']
condition_cols = ['paved_road', 'unpaved_road']
bump_cols      = ['no_speed_bump', 'speed_bump_asphalt', 'speed_bump_cobblestone']
rough_l_cols   = ['good_road_left', 'regular_road_left', 'bad_road_left']
rough_r_cols   = ['good_road_right','regular_road_right','bad_road_right']

labels['surface_type']      = labels[surface_cols].idxmax(axis=1)
labels['surface_condition'] = labels[condition_cols].idxmax(axis=1)
labels['speed_bump']        = labels[bump_cols].idxmax(axis=1)
labels['roughness_left']    = labels[rough_l_cols].idxmax(axis=1)
labels['roughness_right']   = labels[rough_r_cols].idxmax(axis=1)

labels_cat = labels.drop(columns=surface_cols + condition_cols + bump_cols + rough_l_cols + rough_r_cols)

print(left.columns)
print(labels_cat.columns)



Index(['timestamp', 'acc_x_dashboard', 'acc_y_dashboard', 'acc_z_dashboard',
       'acc_x_above_suspension', 'acc_y_above_suspension',
       'acc_z_above_suspension', 'acc_x_below_suspension',
       'acc_y_below_suspension', 'acc_z_below_suspension', 'gyro_x_dashboard',
       'gyro_y_dashboard', 'gyro_z_dashboard', 'gyro_x_above_suspension',
       'gyro_y_above_suspension', 'gyro_z_above_suspension',
       'gyro_x_below_suspension', 'gyro_y_below_suspension',
       'gyro_z_below_suspension', 'mag_x_dashboard', 'mag_y_dashboard',
       'mag_z_dashboard', 'mag_x_above_suspension', 'mag_y_above_suspension',
       'mag_z_above_suspension', 'temp_dashboard', 'temp_above_suspension',
       'temp_below_suspension', 'timestamp_gps', 'latitude', 'longitude',
       'speed'],
      dtype='object')
Index(['surface_type', 'surface_condition', 'speed_bump', 'roughness_left',
       'roughness_right'],
      dtype='object')


## 3. Merge Categorical Labels Into Sensor Data

In [37]:
left = left.reset_index(drop=True)
right = right.reset_index(drop=True)
labels_cat = labels_cat.reset_index(drop=True)

left_labeled = pd.concat([left, labels_cat], axis=1)
right_labeled = pd.concat([right, labels_cat], axis=1)

print(left_labeled[['timestamp', 'surface_type', 'speed_bump']].head())


      timestamp  surface_type     speed_bump
0  1.577219e+09  asphalt_road  no_speed_bump
1  1.577219e+09  asphalt_road  no_speed_bump
2  1.577219e+09  asphalt_road  no_speed_bump
3  1.577219e+09  asphalt_road  no_speed_bump
4  1.577219e+09  asphalt_road  no_speed_bump


## 4. 

In [ ]:
class RoadData:
    def __init__(self, df: pd.DataFrame, target_col: str, test_size=0.2):
        self.df = df.copy()
        self.target_col = target_col

        self.feature_cols = [col for col in df.columns
            if col not in ['timestamp','timestamp_gps','latitude','longitude',
                    'surface_type','surface_condition','speed_bump',
                    'roughness_left','roughness_right']
        ]

        self.classes, self.df[target_col] = np.unique(self.df[target_col], return_inverse=True)

        x = self.df[self.feature_cols]
        y = self.df[target_col]
        self.x_train, self.x_test, self.y_train, self.y_test = train_test_split(x, y, test_size=test_size, random_state=42)


class Classifier:
    def __init__(self, x_train, y_train, x_test, y_test, n_trials=10):
        self.x_train = x_train
        self.y_train = y_train
        self.x_test  = x_test
        self.y_test  = y_test
        self.study = optuna.create_study(direction="maximize")
        self.study.optimize(self._optimize, n_trials=n_trials)

        best = self.study.best_params
        self.model = RandomForestClassifier(
            n_estimators=best['n_estimators'],
            max_depth=best['max_depth'],
            criterion=best['criterion'],
            n_jobs=-1,
            random_state=42
        )
        self.model.fit(self.x_train, self.y_train)

    def _optimize(self, trial):
        criterion = trial.suggest_categorical("criterion", ["gini", "entropy"])
        n_estimators = trial.suggest_int("n_estimators", 10, 100)
        max_depth = trial.suggest_int("max_depth", 1, 20)

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            criterion=criterion,
            n_jobs=-1,
            random_state=42
        )

        cv = KFold(n_splits=3, shuffle=True, random_state=42)
        scores = []

        for tr_idx, val_idx in cv.split(self.x_train):
            model.fit(self.x_train.iloc[tr_idx], self.y_train.iloc[tr_idx])
            scores.append(model.score(self.x_train.iloc[val_idx], self.y_train.iloc[val_idx]))
        return np.mean(scores)
    

rd = RoadData(left_labeled, 'surface_type')
display(rd.df.head())

rc = Classifier(rd.x_train, rd.y_train, rd.x_test, rd.y_test)

y_pred = rc.model.predict(rd.x_test)
print(classification_report(rd.y_test, y_pred, target_names=rd.classes))


,timestamp,acc_x_dashboard,acc_y_dashboard,acc_z_dashboard,acc_x_above_suspension,acc_y_above_suspension,acc_z_above_suspension,acc_x_below_suspension,acc_y_below_suspension,acc_z_below_suspension,...,temp_below_suspension,timestamp_gps,latitude,longitude,speed,surface_type,surface_condition,speed_bump,roughness_left,roughness_right
0,1.577219e+09,0.365116,0.167893,9.793961,0.327626,0.172733,9.781861,0.024797,0.172611,9.793824,...,31.782640,1.577219e+09,-27.717841,-51.098865,0.009128,0,paved_road,no_speed_bump,good_road_left,good_road_right
1,1.577219e+09,0.392649,0.176273,9.771216,0.381496,0.189492,9.699261,0.024797,0.194158,9.842905,...,31.782640,1.577219e+09,-27.717841,-51.098865,0.009128,0,paved_road,no_speed_bump,good_road_left,good_road_right
2,1.577219e+09,0.409408,0.181062,9.732909,0.283333,0.182310,9.807000,0.003249,0.227677,9.888395,...,31.926408,1.577219e+09,-27.717841,-51.098865,0.009128,0,paved_road,no_speed_bump,good_road_left,good_road_right
3,1.577219e+09,0.371101,0.164302,9.749668,0.314458,0.230194,9.739963,0.005643,0.172611,9.871635,...,31.926408,1.577219e+09,-27.717841,-51.098865,0.009128,0,paved_road,no_speed_bump,good_road_left,good_road_right
4,1.577219e+09,0.390255,0.159514,9.869378,0.344385,0.202660,9.762708,0.005643,0.200144,9.860862,...,31.830563,1.577219e+09,-27.717841,-51.098865,0.009128,0,paved_road,no_speed_bump,good_road_left,good_road_right


[I 2025-05-14 12:11:36,607] A new study created in memory with name: no-name-f183d86a-0bcc-426a-9b60-6fa9f87337c7
[I 2025-05-14 12:11:38,497] Trial 0 finished with value: 0.9905318098350343 and parameters: {'criterion': 'entropy', 'n_estimators': 49, 'max_depth': 12}. Best is trial 0 with value: 0.9905318098350343.
[I 2025-05-14 12:11:39,746] Trial 1 finished with value: 0.9834328343563041 and parameters: {'criterion': 'gini', 'n_estimators': 33, 'max_depth': 9}. Best is trial 0 with value: 0.9905318098350343.
[I 2025-05-14 12:11:41,013] Trial 2 finished with value: 0.9730968254783914 and parameters: {'criterion': 'gini', 'n_estimators': 48, 'max_depth': 7}. Best is trial 0 with value: 0.9905318098350343.
[I 2025-05-14 12:11:42,179] Trial 3 finished with value: 0.9597927574933888 and parameters: {'criterion': 'entropy', 'n_estimators': 45, 'max_depth': 6}. Best is trial 0 with value: 0.9905318098350343.
[I 2025-05-14 12:11:45,478] Trial 4 finished with value: 0.9925538888569064 and par

'                  precision    recall  f1-score   support\n\n    asphalt_road       1.00      1.00      1.00     11263\ncobblestone_road       0.99      1.00      1.00     12274\n       dirt_road       1.00      0.98      0.99      5271\n\n        accuracy                           1.00     28808\n       macro avg       1.00      0.99      0.99     28808\n    weighted avg       1.00      1.00      1.00     28808\n'